# WikiArt Art Movement Classification

Train a CNN to classify paintings into art movement categories using the WikiArt dataset.

## 1. Dependencies

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import matplotlib.image as mpimg
import seaborn as sns
import pandas as pd

from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras import regularizers

#Set seed for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

## 2. Auxiliary Functions

In [ ]:
#Count number of image files in each class
def plot_class_distribution(dataset_dir, class_names, title='Class Distribution'):
    counts = []
    image_extensions = {".jpg", ".jpeg", ".png", ".webp"}

    for cls in class_names:
        cls_path = Path(dataset_dir) / cls / cls

        if not cls_path.exists():
            print(f"Warning: missing class folder -> {cls_path}")
            counts.append(0)
            continue

        image_files = [
            p for p in cls_path.iterdir()
            if p.is_file() and p.suffix.lower() in image_extensions]
        counts.append(len(image_files))

    #Sort classes by frequency for easier visual inspection
    sorted_data = sorted(zip(class_names, counts), key=lambda x: x[1], reverse=True)
    class_names_sorted, counts_sorted = zip(*sorted_data)

    print(f'Number of total images: {sum(counts)}')
    print(f'Lowest number of images per class: {min(counts)}')
    print(f'Highest number of images per class: {max(counts)}')

    #Plot horizontal bar chart to highlight class imbalance
    plt.figure(figsize=(10, 6))
    plt.barh(class_names_sorted, counts_sorted)
    plt.xlabel('Number of images')
    plt.title(title)
    plt.tight_layout()
    plt.show()

#Display a few sample images from each selected class
def show_sample_images(dataset_dir, class_names, n_per_class=3):
    fig, axes = plt.subplots(
        len(class_names),
        n_per_class,
        figsize=(n_per_class * 3, len(class_names) * 2.5))

    if len(class_names) == 1:
        axes = np.array([axes])

    image_extensions = {".jpg", ".jpeg", ".png", ".webp"}

    for i, cls in enumerate(class_names):
        cls_path = Path(dataset_dir) / cls / cls

        if not cls_path.exists():
            for j in range(n_per_class):
                axes[i, j].axis('off')
            continue

        image_names = [
            p for p in cls_path.iterdir()
            if p.is_file() and p.suffix.lower() in image_extensions
        ][:n_per_class]

        for j in range(n_per_class):
            axes[i, j].axis('off')

            if j < len(image_names):
                img = mpimg.imread(str(image_names[j]))
                axes[i, j].imshow(img)

            if j == 0:
                axes[i, j].set_title(cls, fontsize=8, fontweight='bold')

    plt.suptitle('Sample Images per Art Movement', fontsize=12)
    plt.tight_layout()
    plt.show()


def process_image_scratch(file_path, label, img_height, img_width):
    #Load image from disk
    img = tf.io.read_file(file_path)

    #Decode image as RGB
    img = tf.image.decode_image(img, channels=3, expand_animations=False)

    #Set static shape for TensorFlow graph tracing
    img.set_shape([None, None, 3])

    #Resize image to target dimensions
    img = tf.image.resize(img, [img_height, img_width])

    #Normalize pixel values to [0, 1]
    img = tf.cast(img, tf.float32) / 255.0
    return img, label


def process_image_tl(file_path, label, img_height, img_width):
    #Load image from disk
    img = tf.io.read_file(file_path)

    #Decode image as RGB
    img = tf.image.decode_image(img, channels=3, expand_animations=False)

    #Set static shape for TensorFlow graph tracing
    img.set_shape([None, None, 3])

    #Resize image to target dimensions
    img = tf.image.resize(img, [img_height, img_width])

    #Keep float32, but do not divide by 255 here
    img = tf.cast(img, tf.float32)
    return img, label

#Build TensorFlow dataset from file paths and labels
def create_dataset(X, y, img_height, img_width, process_fn, batch_size, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))

    if training:
        ds = ds.shuffle(buffer_size=min(len(X), 5000), seed=seed)

    ds = ds.map(
        lambda file_path, label: process_fn(file_path, label, img_height, img_width), num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.cache()
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    return ds

#Plot training and validation curves
def plot_accuracy_loss(history): 
    acc = history.history.get('accuracy', [])
    val_acc = history.history.get('val_accuracy', [])
    loss = history.history.get('loss', [])
    val_loss = history.history.get('val_loss', [])
    epochs_range = range(1, len(loss) + 1)

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    
    if acc:
        plt.plot(epochs_range, acc, label='Training Accuracy')
    if val_acc:
        plt.plot(epochs_range, val_acc, label='Validation Accuracy')
        
    plt.legend(loc='lower right')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    
    if val_loss:
        plt.plot(epochs_range, val_loss, label='Validation Loss')
        
    plt.legend(loc='upper right')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')

    plt.tight_layout()
    plt.show()

def print_class_distribution(y, class_names, split_name):
    counter = Counter(y)
    print(f"\n{split_name} distribution:")

    for class_idx, count in sorted(counter.items()):
        class_name = class_names[class_idx]
        print(f"{class_name}: {count}")

#Generate predictions for evaluation
def get_predictions(model, dataset): 
    y_true = []
    y_pred = []

    for images, batch_labels in dataset:
        preds = model.predict(images, verbose=0)
        pred_classes = np.argmax(preds, axis=1)

        y_true.extend(batch_labels.numpy())
        y_pred.extend(pred_classes)

    return np.array(y_true), np.array(y_pred)

#Plot normalized confusion matrix
def plot_confusion_matrix(y_true, y_pred, class_names): 
    cm = confusion_matrix(y_true, y_pred, normalize='true')

    plt.figure(figsize=(12, 10))
    
    sns.heatmap(
        cm,
        annot=True,
        fmt='.2f',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names)
    
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.title('Normalised Confusion Matrix')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
def plot_f1_scores_by_class(y_true, y_pred, class_names):
    report = classification_report(
        y_true, y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    f1_scores = {cls: report[cls]['f1-score'] for cls in class_names}
    f1_df = pd.DataFrame({
        'Class': list(f1_scores.keys()),
        'F1-score': list(f1_scores.values())
    }).sort_values('F1-score', ascending=False)

    plt.figure(figsize=(10, 6))
    plt.barh(f1_df['Class'], f1_df['F1-score'])
    plt.xlabel('F1-score')
    plt.title('F1-score by Art Movement')
    plt.xlim(0, 1)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
def print_top_confusions(y_true, y_pred, class_names, top_n=5):
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    np.fill_diagonal(cm, 0)

    confusion_pairs = []
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            if i != j:
                confusion_pairs.append((class_names[i], class_names[j], cm[i, j]))

    confusion_pairs = sorted(confusion_pairs, key=lambda x: x[2], reverse=True)

    print(f"Top {top_n} confusions:")
    for true_cls, pred_cls, value in confusion_pairs[:top_n]:
        print(f"{true_cls} → {pred_cls}: {value:.2%}")

## About this Project

WikiArt is an online visual art encyclopaedia containing over 250,000 works spanning centuries of art history. The dataset used here was compiled from WikiArt by organising paintings by their associated art movement label.

**Why is this an interesting deep learning problem?**  
Most image classification benchmarks seen during class involve classes that are visually very distinct. 
Art movement classification is fundamentally different, here they're defined by subtle properties (brushwork, colour palette, compositional style, historical context, etc). 

**How the dataset was built:**  
Labels were assigned by WikiArt's community of contributors and curated by art historians.
This introduces two important challenges: label ambiguity (many paintings sit at the boundary between movements) and class imbalance. 
Both are addressed explicitly in this notebook.

**Scope of this notebook:**  
This workbook works with 10 art movements selected for having enough images to train reliably and for covering a range of visual distinctiveness, going from highly distinct (Japanese_Art vs
Baroque) to genuinely ambiguous (Expressionism vs Primitivism). The full dataset has 13 movements; expanding to all 13 is identified as future work.

## 3. Dataset

**Key characteristics:**
- Variable resolution and aspect ratio as paintings come in all shapes and sizes.
- Single-label classification (each image belongs to exactly one movement).
- Significant class imbalance
- Label ambiguity: the boundaries between movements are debated even among experts. The model must learn subtle stylistic differences that humans also find difficult to articulate.

Unlike simpler datasets the present work presents a bigger problem as art movements are defined by style, texture, colour palette and/or brushwork. These are features that are global and subtle, as such, a CNN must learn not just what is in a painting but how it was made. 

#### 3.1 Class Selection

In [ ]:
source_dir = Path(r"C:\Users\Inês\.cache\kagglehub\datasets\sivarazadi\wikiart-art-movementsstyles\versions\2")

selected_classes = [
    'Art_Nouveau',
    'Baroque',
    'Expressionism',
    'Japanese_Art',
    'Neoclassicism',
    'Primitivism',
    'Realism',
    'Renaissance',
    'Rococo',
    'Romanticism'
]

dataset_dir = source_dir
num_classes = len(selected_classes)

image_extensions = {".jpg", ".jpeg", ".png", ".webp"}

#Build list of image file paths and integer labels
file_paths = []
labels = []

for label_idx, cls in enumerate(selected_classes):
    class_dir = source_dir / cls / cls

    if not class_dir.exists():
        print(f"Missing source folder for class: {cls}")
        continue

    class_files = [str(p) for p in class_dir.iterdir() if p.is_file() and p.suffix.lower() in image_extensions]

    file_paths.extend(class_files)
    labels.extend([label_idx] * len(class_files))

    print(f"{cls}: {len(class_files)} images")

print(f"\nNumber of classes selected: {num_classes}")

#### 3.2 Class Distribution and Sample Images

Before building any model, it helps to quickly look at the data to confirm images load correctly and labels are meaningful and check the classes with far fewer images that will likely have lower per-class F1-scores, as knowing this in advance prevents misinterpreting model results.

In [ ]:
plot_class_distribution(dataset_dir, selected_classes, title='Images per Art Movement')

In [ ]:
show_sample_images(dataset_dir, selected_classes[:5], n_per_class=3)

#### 3.3 Image Size Distribution

WikiArt images have highly variable resolutions, a scanned medieval manuscript and a photograph of a large canvas have very different pixel dimensions. 
As such, all images must be resized to a fixed size before being fed to the model. 

The histogram below shows the raw size distribution across a sample of images, motivating this preprocessing step.

In [ ]:
#Show original image sizes to motivate the resize decision
sample_sizes = []
for cls in selected_classes[:5]:
    cls_path = Path(dataset_dir) / cls / cls
    files = list(cls_path.iterdir())[:20]
    for f in files:
        try:
            img = mpimg.imread(str(f))
            sample_sizes.append(img.shape[:2])
        except:
            pass

heights = [s[0] for s in sample_sizes]
widths  = [s[1] for s in sample_sizes]

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(heights, bins=20, color='steelblue')
plt.xlabel('Height (px)')
plt.title('Original image heights')

plt.subplot(1, 2, 2)
plt.hist(widths, bins=20, color='steelblue')
plt.xlabel('Width (px)')
plt.title('Original image widths')

plt.suptitle('Raw image size distribution (sample)')
plt.tight_layout()
plt.show()

print(f"Height range: {min(heights)}px – {max(heights)}px")
print(f"Width range:  {min(widths)}px – {max(widths)}px")

## 4. Evaluation Protocol and Data Splits

The dataset is divided into the below non-overlapping subsets.

| Split | Size | Purpose |
|---|---|---|
| Training | 70% | Used to update model weights via gradient descent |
| Validation | 15% | Monitors overfitting during training; never updates weights |
| Test | 15% | Used only once at the very end to report final performance |

**Why hold-out and not K-fold?**  
K-fold cross-validation would be statistically more robust, however, it multiplies training time by K and with around 42,000 images, hold-out is the practical choice. 
Additionally, the validation and test sets are large enough (around 6,000 images each, across 10 classes) that the accuracy estimate has low variance.

**Why stratified splitting?**  
The component `stratify=labels` ensures that the class distribution in each split mirrors the original dataset and is especially important here because the dataset is imbalanced.

**The test set rule:**  
The test set is evaluated exactly once, after all hyperparameter decisions have been made using only the validation set. 
Using test set results to guide model choices constitutes data leakage and produces overoptimistic performance estimates.

#### 4.1 Configuration
The main hyperparameters used throughout the notebook are defined here.

Two input sizes are used:
- For CNNs trained from scratch, images are resized to 96 × 96. This reduces computational cost and allows faster experimentation on a local machine.
- For transfer learning with EfficientNetB0, images are resized to 160 × 160 to match the expected input size of the pre-trained model.

Two batch sizes are used: **32** for the scratch models (96×96 images fit comfortably) and **16** for transfer learning.

In [ ]:
img_height_small = 96
img_width_small = 96

img_height_tl = 160
img_width_tl = 160

batch_size_small = 32
batch_size_tl = 16

seed = 42

#20% of training/val data used for all scratch models
TRAIN_FRACTION = 0.20

#200 images per class for fast hyperparameter search only
N_PER_CLASS_TUNE = 200

#### 4.2 Building the Pipeline

Images are loaded and preprocessed using a custom `tf.data.Dataset` pipeline rather than loading everything into memory at once. 

This approach:
- Loads and decodes images on demand (lazy evaluation)
- Applies resizing and normalisation as part of the graph
- Uses `prefetch` so the CPU prepares the next batch while the model trains on the current one
- - Uses `cache` to store processed images in memory after the first epoch, eliminating repeated disk reads and significantly speeding up subsequent epochs

Two preprocessing functions are used:
- For CNNs trained from scratch, pixel values are normalized to [0, 1]
- For transfer learning, images are kept as float32 and passed through the model-specific preprocessing function required by EfficientNet

Additionally, a data cleaning step was performed to remove invalid or corrupted image files that could not be decoded by TensorFlow.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(file_paths, labels, test_size=0.30, stratify=labels, random_state=seed)
X_val, X_test, y_val, y_test = train_test_split( X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=seed)

#Use 20% of training and validation for all scratch model experiments
X_train, _, y_train, _ = train_test_split(X_train, y_train, train_size=TRAIN_FRACTION, stratify=y_train, random_state=seed)
X_val, _, y_val, _ = train_test_split( X_val, y_val, train_size=TRAIN_FRACTION, stratify=y_val, random_state=seed)

print(f"Scratch models  — Training: {len(X_train)}, Validation: {len(X_val)}")
print(f"Test set (full) — {len(X_test)} images")

class_weights = compute_class_weight( class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

#All models use the same subset (30%)
ds_train_small = create_dataset(X_train, y_train, img_height_small, img_width_small, process_fn=process_image_scratch, batch_size=batch_size_small, training=True)
ds_val_small = create_dataset(X_val, y_val, img_height_small, img_width_small, process_fn=process_image_scratch, batch_size=batch_size_small, training=False)
ds_test_small = create_dataset(X_test, y_test, img_height_small, img_width_small, process_fn=process_image_scratch, batch_size=batch_size_small, training=False)

#TL needs 160×160
ds_train_tl = create_dataset(X_train, y_train, img_height_tl, img_width_tl, process_fn=process_image_tl, batch_size=batch_size_tl, training=True)
ds_val_tl = create_dataset(X_val, y_val, img_height_tl, img_width_tl, process_fn=process_image_tl, batch_size=batch_size_tl, training=False)
ds_test_tl = create_dataset(X_test, y_test, img_height_tl, img_width_tl, process_fn=process_image_tl, batch_size=batch_size_tl, training=False)

In [ ]:
#Tiny tuning dataset, used only in the dropout and learning rate search loops
X_tune, y_tune = [], []
X_val_tune, y_val_tune = [], []

for class_idx in range(num_classes):
    #Training subset
    idx_train = [i for i, y in enumerate(y_train) if y == class_idx]
    idx_train = idx_train[:N_PER_CLASS_TUNE]
    X_tune.extend([X_train[i] for i in idx_train])
    y_tune.extend([y_train[i] for i in idx_train])

    #Validation subset
    idx_val = [i for i, y in enumerate(y_val) if y == class_idx]
    idx_val = idx_val[:N_PER_CLASS_TUNE]
    X_val_tune.extend([X_val[i] for i in idx_val])
    y_val_tune.extend([y_val[i] for i in idx_val])

ds_train_tune = create_dataset(X_tune, y_tune, img_height_small, img_width_small, process_fn=process_image_scratch, batch_size=batch_size_small, training=True)
ds_val_tune = create_dataset(X_val_tune, y_val_tune, img_height_small, img_width_small, process_fn=process_image_scratch, batch_size=batch_size_small, training=False)

print(f"Tuning set: {len(X_tune)} train ({N_PER_CLASS_TUNE}/class), "f"{len(X_val_tune)} val ({N_PER_CLASS_TUNE}/class)")

#### 4.2.1 Distribution Analysis

The stratified split preserves the class distribution across all three subsets.
The counts below confirm that the imbalance present in the original dataset (Romanticism with around 6,800 images vs Primitivism with around 1,300) is reflected proportionally in training, validation, and test where no class is accidentally over or under-represented in any split.

The class weights printed below are used during training to compensate for this imbalance. 
Classes with fewer images receive proportionally higher weights, so the model is penalised more heavily for misclassifying a rare class than a frequent one.

In [ ]:
print_class_distribution(y_train, selected_classes, "Train")
print_class_distribution(y_val, selected_classes, "Validation")
print_class_distribution(y_test, selected_classes, "Test")

In [ ]:
for i, cls in enumerate(selected_classes):
    print(f"{cls}: weight={class_weight_dict[i]:.2f}")

#### 4.3 Verify Output

A quick sanity check to confirm the pipeline is producing correctly shaped before any model is built.

In [ ]:
#Verify shape and normalization for the small-image pipeline
for images, batch_labels in ds_train_tl.take(1):
    print(f"TL image batch shape: {images.shape}")
    print(f"TL label batch shape: {batch_labels.shape}")
    print(f"TL pixel value range: [{images.numpy().min():.3f}, {images.numpy().max():.3f}]")

#### 4.4 Data Augmentation

Data augmentation artificially increases the effective diversity of the training set by applying small random transformations to each image (only during the training).
As such, the validation and test images are kept unchanged, since they should reflect the original data distribution as closely as possible as we want to evaluate on unmodified images.

The transformations used here are deliberately conservative:

- **Small random rotation:** helps simulate slight differences in how paintings may have been photographed or scanned.
- **Random zoom:** makes the model less sensitive to framing and crop differences.
- **Random brightness:** helps account for variation in lighting, scan quality, and digitization conditions.

More aggressive transformations were avoided since they can distort important features in paintings

In [ ]:
data_augmentation_scratch = keras.Sequential([
    layers.RandomRotation(0.03, fill_mode='reflect'),
    layers.RandomZoom(0.1, fill_mode='reflect'),
    #layers.RandomBrightness(0.1, value_range=(0, 1)),
], name='data_augmentation_scratch')

data_augmentation_tl = keras.Sequential([
    layers.RandomRotation(0.05, fill_mode='reflect'),
    layers.RandomZoom(0.15, fill_mode='reflect'),
], name='data_augmentation_tl')

for images, _ in ds_train_small.take(1):
    plt.figure(figsize=(12, 4))
    first_image = images[0]

    plt.subplot(1, 5, 1)
    plt.imshow(first_image.numpy())
    plt.title('Original')
    plt.axis('off')

    for i in range(4):
        augmented = data_augmentation_scratch(tf.expand_dims(first_image, 0), training=True)
        aug_img = tf.clip_by_value(augmented[0], 0.0, 1.0).numpy()

        plt.subplot(1, 5, i + 2)
        plt.imshow(aug_img)
        plt.title(f'Aug {i+1}')
        plt.axis('off')

    plt.suptitle('Effect of Data Augmentation')
    plt.tight_layout()
    plt.show()

In [ ]:
#Show a single image through the full preprocessing pipeline
sample_path = X_train[10]
raw_img = mpimg.imread(sample_path)

img_tensor = tf.image.resize(
    tf.cast(raw_img, tf.float32),
    [img_height_small, img_width_small]
) / 255.0

fig, axes = plt.subplots(1, 3, figsize=(10, 3))

axes[0].imshow(raw_img)
axes[0].set_title(f'Original\n{raw_img.shape[0]}×{raw_img.shape[1]}px')
axes[0].axis('off')

axes[1].imshow(img_tensor.numpy())
axes[1].set_title(f'After resize\n{img_height_small}×{img_width_small}px')
axes[1].axis('off')

aug = data_augmentation_scratch(tf.expand_dims(img_tensor, 0), training=True)
axes[2].imshow(tf.clip_by_value(aug[0], 0, 1).numpy())
axes[2].set_title('After augmentation')
axes[2].axis('off')

plt.suptitle('Preprocessing pipeline — single image example')
plt.tight_layout()
plt.show()

### 4.5 Evaluation Metrics

Three complementary metrics are reported throughout this notebook:

**Accuracy** is the primary training metric as the proportion of predictions that are correct. 
It is easy to monitor during training and directly answers "how often is the model right?". 
However, with class imbalance a model can achieve misleadingly high accuracy by over-predicting frequent classes. 
A model that always predicts Romanticism would achieve ~17% accuracy without learning anything useful.

**Top-3 accuracy** counts a prediction as correct if the true class appears among the model's top 3 predicted classes. 
For a 10-class problem this is a more lenient measure but it is informative for art classification because a painting that straddles two movements might genuinely be ambiguous. 
If the model places the true label in second place, top-3 accuracy captures that partial correctness.

**F1-score per class** is reported at evaluation time. 
F1 is the harmonic mean of precision and recall and is more informative than accuracy when classes are imbalanced.
A class with very few examples may have low F1 even if overall accuracy is high and this is important to know before drawing conclusions about model performance.

**Confusion matrix** shows which classes are confused with each other. 
For art movements this is particularly valuable: systematic confusion between Expressionism and Primitivism tells us something meaningful about what the model has and has not learned, and aligns with what a human art historian would also find difficult.

**Why not use weighted accuracy or macro-F1 as the primary metric?**  
Accuracy is used as the primary training signal because it is the most direct reflection of cross-entropy loss minimisation and is easy to interpret live during training. 
The richer metrics (F1 per class, confusion matrix) are computed once at the end on the test set, where they provide the most actionable information.

## 5. Modelling

The modelling section follows: build a simple baseline first then increase complexity until the model overfits then apply regularisation to recover generalisation. 
Each step is a deliberate diagnostic and not just an attempt to improve accuracy.


#### 5.1 Baseline Model

The baseline uses two convolutional blocks followed by a `Flatten` layer and a small dense classifier. 
This architecture serves as a known reference point.

The goal at this stage is not performance, it is to:
1. Verify the full pipeline works end to end (data to model to predictions)
2. Establish a floor that every subsequent model should clearly beat
3. Confirm that even a minimal architecture can learn something above random chance (10% for 10 classes)

**Architecture decisions:**

`Conv2D` with `relu` activation is used in the convolutional blocks. ReLU is the standard choice for hidden layers as it is fast to compute and avoids the vanishing gradient problem of sigmoid.

`MaxPooling2D` after each convolutional block progressively reduces the spatial dimensions, forcing the network to retain only the most prominent features. 
This also reduces the number of parameters in subsequent layers.

`Flatten` converts the final feature maps into a 1D vector before the dense classifier.
This is the most transparent way to connect convolutional features to a classifier. The trade-off compared to `GlobalAveragePooling2D` (used in later models) is a larger dense layer which is acceptable
here because the baseline is intentionally small.

`softmax` in the output layer produces a probability distribution over the 10 classes, which is the correct choice for multi-class single-label classification. 

`SparseCategoricalCrossentropy` is used as the loss function because labels are integers (0–9) rather than one-hot encoded vectors. 
This avoids an unnecessary encoding step and is directly equivalent to `CategoricalCrossentropy` on one-hot labels.

`Adam` is used as the optimiser as it adapts the learning rate per parameter and is the most widely used optimiser for neural networks. 
On the other hand, a fixed learning rate with standard SGD would require more tuning to achieve comparable results.

The baseline goes directly from Flatten to the output layer. 
This creates a linear  classifier on top of the convolutional features with the minimal possible head. 
The goal is for the baseline to learn something useful without overfitting, so that the contrast with the complex model in the next section is clear. 
Adding a Dense layer between Flatten and output at this dataset size introduces enough capacity to overfit before we deliberately intend it to.

In [ ]:
baseline_model = Sequential([
    layers.Input(shape=(img_height_small, img_width_small, 3)),

    #16 filters & minimal feature extraction
    layers.Conv2D(16, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(),  # 96 to 48

    #32 filters & slightly richer representations
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(),  # 48 to 24

    
    layers.Flatten(), #Flatten: 24×24×32 = 18,432 values

    # No intermediate Dense & go directly to output
    layers.Dense(num_classes, activation='softmax', name='output')
], name='baseline')

baseline_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top_3_accuracy')])

baseline_model.summary()

In [ ]:
epochs = 3

history_baseline = baseline_model.fit(
    ds_train_small,
    validation_data=ds_val_small,
    epochs=epochs,
    class_weight=class_weight_dict)

In [ ]:
plot_accuracy_loss(history_baseline)

A random classifier, with 10 classes, would achieve approximately 10% (1/10 classes) accuracy. 
On the other hand, a majority-class baseline would achieve a higher accuracy due to class imbalance, as it always predicts the most frequent class.

The baseline reaches validation accuracy clearly above random chance, which shows that the model is already learning useful visual patterns from the paintings. At the same time, the training and validation curves remain relatively close, suggesting that this architecture is still too limited to strongly overfit the data. In other words, it works as a baseline, but it is probably not expressive enough to capture the full complexity of the problem.

#### 5.2 Overfitting Model

To deliberately induce overfitting, model capacity is increased across three dimensions simultaneously: depth, width, and the size of the classifier head.

Compared to the baseline, this model has roughly 20× more parameters. The goal is not a better model, it is to confirm that the architecture can memorise the training data, which proves the dataset contains learnable signal before we attempt to regularise.

**Why double Conv2D layers per block instead of single layers?**  
The baseline and the course examples use one Conv2D per block. Now, it stacks two Conv2D layers before each MaxPooling. 
This patters of two consecutive convolutions at the same resolution allows the network to compose more complex features before spatial downsampling. 
A single Conv2D sees a 3×3 neighbourhood and two stacked Conv2D layers effectively see a 5×5 neighbourhood with fewer parameters than a single 5×5 kernel, and with an extra non-linearity between them.

**Why start with 32 filters instead of 16?**  
The baseline uses 16 filters in the first block which is enough for a minimal model. 
Now it starts at 32 and double at each block (32 to 64 to 128). This leads to an increase of filter count as spatial resolution decreases: as MaxPooling compresses height and width, more filters are needed to preserve representational capacity.

**Why GlobalAveragePooling2D here instead of Flatten?**  
The baseline uses Flatten as shown in the course examples. Here there's a switch to GlobalAveragePooling2D for the complex model (and all subsequent models). 
After three MaxPooling operations on 96×96 input, the feature maps are 12×12×128. 
Flattening this would produce a vector of 18,432 values feeding into Dense(256) in that single connection alone, which would dominate training time and make overfitting even harder to control later. 
GlobalAveragePooling2D reduces the same feature maps to a vector of 128 values, making the model trainable on a CPU and allowing the overfitting to come from the convolutional capacity rather than from a massive
dense layer.

**Why two Dense layers (256 to 128) instead of three decreasing ones?**  
Using Dense(256) to Dense(128) aims to increase the classifier capacity proportionally to the larger convolutional backbone. 
Three layers would add more depth without meaningfully more capacity at this scale, and would slow training without a clear benefit for a 10-class problem.

**Class weights are not applied here.**  
The purpose of this model is to overfit the training data as clearly as possible. Applying class weights would partially counteract that by rebalancing gradients, making it harder to observe the overfitting signal. 
Class weights are reintroduced in the regularised model.

In [ ]:
#Larger CNN designed to overfit the training set

overfit_model = Sequential([
    layers.Input(shape=(img_height_small, img_width_small, 3)),

    #two Conv2D at 32 filters before pooling
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(),  #96 to 48

    #double filters as spatial resolution decreases
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(),  #48 to 24

    #128 filters & high-level feature extraction
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(),  # 24 to 12

    #Flatten instead of GAP to maximise parameters
    layers.Flatten(),
    layers.Dense(128, activation='relu'), #12×12×128 = 18,432 values & 10M params in this connection alone
    layers.Dense(num_classes, activation='softmax', name='output')], name='overfit_model')

overfit_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top_3_accuracy')])


overfit_model.summary()

In [ ]:
epochs = 6

history_overfit = overfit_model.fit(
    ds_train_small,
    validation_data=ds_val_small,
    epochs=epochs)

In [ ]:
plot_accuracy_loss(history_overfit)

## 6. Regularised Model

After confirming overfitting in the previous model, regularisation is applied to improve generalisation. 

The architecture is intentionally **identical** to the overfit model with the same double conv blocks, Flatten, and Dense(512) + Dense(256) head. 
This makes the comparison clean: any improvement in generalisation can be attributed entirely to the regularisation techniques, not to a smaller architecture.

Three complementary techniques are combined:

**Data Augmentation** is added here but was absent from the overfit model. 
By showing slightly modified versions of each painting during training, the model cannot memorise exact pixel patterns and must instead learn broader stylistic features. It was excluded from the overfit model so that overfitting could be observed as clearly as possible.

**Dropout** randomly sets a fraction of activations to zero during each training step.
This prevents co-adaptation between neurons — no single neuron can rely on specific others always being present, forcing each to independently learn useful features.
Dropout is only active during training; at inference time all neurons contribute and outputs are scaled by `(1 - dropout_rate)`.

The dropout rate is a hyperparameter that requires tuning. 
Too low and overfitting persists and too high and the model loses too much capacity and underfits. 
Three configurations are tested here to find the best balance:

| Config | Conv Dropout | Dense Dropout |
|---|---|---|
| Low | 0.1 | 0.2 |
| Mid | 0.2 | 0.4 |
| High | 0.3 | 0.5 |

Conv and dense dropout are kept asymmetric across all configs. Convolutional features are spatially redundant so they tolerate lighter dropout, while dense layers are the primary site of memorisation and receive stronger regularisation.

**L2 Weight Regularisation** adds a penalty proportional to the squared magnitude of each weight. 
This discourages large weights, which are a symptom of overfitting. 
The coefficient `1e-5` is deliberately small so it nudges weights without preventing learning.

**Lower learning rate (3e-4)** compared to the default Adam `1e-3`. 
With regularisation already constraining updates, a slightly lower rate produces more stable convergence and works better with `ReduceLROnPlateau`.

The best dropout configuration is selected based on validation accuracy and used as `regularized_model` in the final comparison.

To address class imbalance, class weights were computed and applied during training. This penalizes errors in underrepresented classes more heavily and helps the model learn more balanced decision boundaries.

**Hyperparameter tuning:**  
Two hyperparameters are searched systematically. First, dropout rates are compared across three configurations with the default learning rate. Then, with the best dropout fixed, three learning rates are compared. This sequential search is more efficient than a full grid search (9 combinations) while still covering the most impactful parameters.

The winning configuration is identified on the subsample and then retrained on the full dataset to produce the final regularised model.

**Note on search speed:**  
To make this search practical, each configuration is evaluated on a stratified 30% subsample of the training data (controlled by `USE_SUBSAMPLE`). 
The relative ranking of configurations on the subsample is reliable for identifying  the best combination. 
The winning configuration is then retrained on the full dataset in the final step.

In [ ]:
callbacks_regularized_tuning = [
    #Stop training when validation loss stops improving
    #tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True ),
    
    #Reduce learning rate when validation loss plateaus
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',  factor=0.3, patience=1, min_lr=1e-6),
    #tf.keras.callbacks.ModelCheckpoint('best_regularized.keras', monitor='val_loss', save_best_only=True)
    ]

In [ ]:
def build_regularized_model(dropout_conv, dropout_dense, learning_rate=3e-4, name='regularized'):
    model = Sequential([
        layers.Input(shape=(img_height_small, img_width_small, 3)),

        #Augmentation only active during training
        data_augmentation_scratch,

        #Same architecture as overfit_model + regularisation
        layers.Conv2D(32, 3, padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.Conv2D(32, 3, padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.MaxPooling2D(),
        layers.Dropout(dropout_conv),

        layers.Conv2D(64, 3, padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.Conv2D(64, 3, padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.MaxPooling2D(),
        layers.Dropout(dropout_conv),

        layers.Conv2D(128, 3, padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dropout(dropout_dense),
        layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dense(num_classes, activation='softmax')], name=name)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy',tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top_3_accuracy')])
    
    return model

In [ ]:
dropout_configs = [
    (0.1, 0.2, 'regularized_low_dropout'),
    (0.2, 0.4, 'regularized_mid_dropout'),
    (0.3, 0.5, 'regularized_high_dropout'),]

dropout_histories = {}

for dropout_conv, dropout_dense, name in dropout_configs:
    print(f"\nTraining: {name} (conv={dropout_conv}, dense={dropout_dense})")
    model = build_regularized_model(dropout_conv, dropout_dense, name=name)
    
    history = model.fit(
        ds_train_tune,      
        validation_data=ds_val_tune,  
        epochs=3,
        callbacks=callbacks_regularized_tuning,
        class_weight=class_weight_dict,
        verbose=1)
    
    dropout_histories[name] = {
        'history': history,
        'model': model,
        'dropout_conv': dropout_conv,
        'dropout_dense': dropout_dense}

In [ ]:
#Select best dropout config based on validation accuracy
best_dropout_name = max(dropout_histories, key=lambda k: max(dropout_histories[k]['history'].history['val_accuracy']))
regularized_model = dropout_histories[best_dropout_name]['model']
history_regularized = dropout_histories[best_dropout_name]['history']

print(f"Selected: {best_dropout_name}")

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
for name, data in dropout_histories.items():
    val_acc = data['history'].history['val_accuracy']
    label = f"conv={data['dropout_conv']}, dense={data['dropout_dense']}"
    plt.plot(range(1, len(val_acc) + 1), val_acc, label=label)
plt.title('Validation Accuracy — Dropout Comparison')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
for name, data in dropout_histories.items():
    val_loss = data['history'].history['val_loss']
    label = f"conv={data['dropout_conv']}, dense={data['dropout_dense']}"
    plt.plot(range(1, len(val_loss) + 1), val_loss, label=label)
plt.title('Validation Loss — Dropout Comparison')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
#With the best dropout fixed, test three learning rates
best_dropout_conv = dropout_histories[best_dropout_name]['dropout_conv']
best_dropout_dense = dropout_histories[best_dropout_name]['dropout_dense']

lr_configs = [1e-3, 3e-4, 1e-4]
lr_histories = {}

for lr in lr_configs:
    print(f"\nTraining with lr={lr}")
    model = build_regularized_model(
        dropout_conv=best_dropout_conv,
        dropout_dense=best_dropout_dense,
        learning_rate=lr,
        name=f'regularized_lr{lr}')

    history = model.fit(
        ds_train_tune, 
        validation_data=ds_val_tune, 
        epochs=3,
        callbacks=callbacks_regularized_tuning,
        class_weight=class_weight_dict,
        verbose=1)

    lr_histories[lr] = {'history': history, 'model': model}


In [ ]:
#Comparison plot
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
for lr, data in lr_histories.items():
    val_acc = data['history'].history['val_accuracy']
    plt.plot(range(1, len(val_acc) + 1), val_acc, label=f'lr={lr}')
plt.title('Validation Accuracy — Learning Rate Comparison')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
for lr, data in lr_histories.items():
    val_loss = data['history'].history['val_loss']
    plt.plot(range(1, len(val_loss) + 1), val_loss, label=f'lr={lr}')
plt.title('Validation Loss — Learning Rate Comparison')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#Select best learning rate by validation accuracy
best_lr = max(lr_histories, key=lambda k: max(lr_histories[k]['history'].history['val_accuracy']))

#The model trained with best_lr IS the final regularized model
regularized_model = lr_histories[best_lr]['model']
history_regularized = lr_histories[best_lr]['history']

print(f"Best learning rate: {best_lr}")
print(f"Best dropout_conv:  {best_dropout_conv}")
print(f"Best dropout_dense: {best_dropout_dense}")
print(f"\nFinal regularized model selected — no retraining needed.")

In [ ]:
plot_accuracy_loss(history_regularized)

### 7. Transfer Learning with EfficientNetB0

The previous models were trained from scratch where all weights were randomly initialised and learned entirely from the WikiArt data. 
Transfer learning takes a different approach: reuse a model pre-trained on ImageNet and adapt it to the present problem.

**Why does this work?**  
The early layers of any CNN learn generic visual features, such as, edges, textures, colour gradients, and simple shapes. 
These features are useful regardless of the domain.
Even though ImageNet contains photographs of everyday objects (cats, cars, furniture) and WikiArt contains paintings, the low-level visual vocabulary is shared: a brushstroke has edges and textures just like a photograph does. The model does not need to relearn how to detect these from scratch.

The transfer learning strategy applied was to freeze the entire base and only train the new classification head.

**Why EfficientNetB0?**  
EfficientNetB0 was designed to maximise accuracy per unit of compute. 
It achieves strong ImageNet performance with far fewer parameters than VGG16 or ResNet50. 
The B0 variant is the smallest in the EfficientNet family, the right trade-off for this project's computational constraints. 
What matters here is demonstrating that pre-trained weights transfer usefully to a new domain.

**Why 160×160 input instead of 224×224?**  
EfficientNetB0 was designed for 224×224 but accepts any input size above 32×32. 
Using 160×160 reduces the memory and compute cost significantly where each image has around 50% fewer pixels while still giving the convolutional base enough resolution to extract meaningful style features. 
This was a practical compromise that needed to be implemented in the present project.

**Dropout in the classification head:**  
Rather than choosing a dropout rate arbitrarily, the best rate identified during the regularised model experiments is reused here. 
This is a deliberate decision: the dropout tuning already established which rate best balances regularisation and capacity for this dataset, and there is no reason to expect a different optimum in the dense layers of the transfer learning head.


**Why `base_model(x, training=False)`?**  
EfficientNetB0 contains BatchNormalisation layers. With `training=False`, these layers use their pre-trained running statistics (mean and variance from ImageNet) rather than computing new statistics from the current batch. This is important when the base is frozen where if BatchNorm updated its statistics on WikiArt batches, the pre-trained features would gradually drift, undermining the point of freezing the base.

**Why `efficientnet.preprocess_input`?**  
EfficientNet was pre-trained with a specific input normalisation. 
Applying the same preprocessing at inference time ensures the pixel distribution the base model receives matches what it was trained on, which is a necessary condition for the pre-trained features to be meaningful.

**Why a lower learning rate (1e-4)?**  
The classification head is randomly initialised — its weights start far from a good solution. 
A high learning rate would cause large gradient updates that could destabilise the BatchNorm statistics in the frozen base even with `training=False`. A lower rate allows the head to converge gradually and is standard practice for transfer learning.

**Callbacks:**  
The same EarlyStopping and ReduceLROnPlateau strategy as the regularised model is used. EfficientNetB0 with a frozen base typically converges within 10–15 epochs on a dataset of this size, so EarlyStopping with patience=4 is appropriate. 
ModelCheckpoint saves the best weights in case validation loss temporarily worsens before recovering.

Input images are passed through the EfficientNet preprocessing function before entering the model as this ensures compatibility with the distribution of the data used during pre-training on ImageNet.

In [ ]:
callbacks_transfer = [
    #Stop training when validation loss stops improving
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True),
    
    #Reduce learning rate when validation loss plateaus
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        min_lr=1e-6),
    
    tf.keras.callbacks.ModelCheckpoint(
        'best_transfer.keras',
        monitor='val_loss',
        save_best_only=True)]

In [ ]:
#Load EfficientNetB0 pre-trained on ImageNet
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(img_height_tl, img_width_tl, 3))

#Freeze convolutional base
base_model.trainable = False

print(f'Base model layers: {len(base_model.layers)}')
print(f'Trainable variables before unfreezing: {len(base_model.trainable_variables)}')

In [ ]:
#Reuse best dropout from regularized model experiments
best_dropout_dense = dropout_histories[best_dropout_name]['dropout_dense']

inputs = tf.keras.Input(shape=(img_height_tl, img_width_tl, 3))

x = data_augmentation_tl(inputs)  # conservative augmentation, no brightness for TL

#EfficientNet-specific normalisation 
x = tf.keras.applications.efficientnet.preprocess_input(x)

#Training=False keeps BatchNorm layers in inference mode
x = base_model(x, training=False)

#Reduce spatial feature maps to a single vector per image
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(best_dropout_dense)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(best_dropout_dense)(x)

outputs = layers.Dense(num_classes, activation='softmax', name='output')(x)

transfer_model = tf.keras.Model(inputs, outputs, name='transfer_efficientnetb0')

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top_3_accuracy')])


transfer_model.summary()

In [ ]:
epochs = 12

history_transfer = transfer_model.fit(
    ds_train_tl,
    validation_data=ds_val_tl,
    epochs=epochs,
    callbacks=callbacks_transfer,
    class_weight=class_weight_dict)

In [ ]:
plot_accuracy_loss(history_transfer)

### 8. Model Comparison

We compare the best validation accuracy achieved by each model across the training process. This gives an overview of how each step in the methodology improved performance.

In [ ]:
best_model = transfer_model

In [ ]:
#Compare the best validation accuracy reached by each model
baseline_best = max(history_baseline.history['val_accuracy'])
overfit_best = max(history_overfit.history['val_accuracy'])
regularized_best = max(history_regularized.history['val_accuracy'])
transfer_best = max(history_transfer.history['val_accuracy'])

print(f"Baseline best val accuracy:     {baseline_best:.4f}")
print(f"Overfit model best val accuracy:{overfit_best:.4f}")
print(f"Regularized best val accuracy:  {regularized_best:.4f}")
print(f"Transfer best val accuracy:     {transfer_best:.4f}")

In [ ]:
#Plot best validation accuracy per model
model_names = ['Baseline', 'Overfit', 'Regularized', 'Transfer']
best_scores = [baseline_best, overfit_best, regularized_best, transfer_best]

plt.figure(figsize=(8, 4))
plt.bar(model_names, best_scores)
plt.ylabel('Best validation accuracy')
plt.title('Model comparison')
plt.ylim(0, 1)
plt.show()

### 9. Evaluation on the Test Set

After all design choices have been made using the training and validation sets, the final step is to evaluate the selected models on the test set.

This is the most important performance estimate in the notebook, because the test set has not been used during model development. The reported results therefore give a more honest measure of how well the model generalises to unseen paintings.

In addition to overall accuracy, the evaluation should also include a confusion matrix and class-level metrics, since some movements are visually much closer to each other than others.

In [ ]:
val_scores = {
    "Baseline": baseline_best,
    "Overfit": overfit_best,
    "Regularized": regularized_best,
    "Transfer": transfer_best
}

best_model_name = max(val_scores, key=val_scores.get)
print("Selected model:", best_model_name)

if best_model_name == "Baseline":
    best_model = baseline_model
    best_test_ds = ds_test_small
elif best_model_name == "Overfit":
    best_model = overfit_model
    best_test_ds = ds_test_small
elif best_model_name == "Regularized":
    best_model = regularized_model
    best_test_ds = ds_test_small
else:
    best_model = transfer_model
    best_test_ds = ds_test_tl

In [ ]:
test_loss, test_acc = best_model.evaluate(best_test_ds, verbose=1)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

In [ ]:
#Evaluate the selected final model on the appropriate test set
y_true, y_pred = get_predictions(best_model, best_test_ds)

print(classification_report(y_true, y_pred, target_names=class_names)) 

In [ ]:
plot_f1_scores_by_class(y_true, y_pred, class_names)

In [ ]:
plot_confusion_matrix(y_true, y_pred, class_names)

In [ ]:
print_top_confusions(y_true, y_pred, class_names, top_n=5)

## 10. Utility of the Model

**Digital museum cataloguing:** Museums digitising large collections often have incomplete metadata. 
An art movement classifier could automatically suggest movement tags for unlabelled works, reducing the manual effort of curators.

**Educational tools:** An application that identifies the art movement of a painting from a photo could be used in museum audio guides, art history courses, or consumer apps for gallery visitors. 
The top-3 accuracy metric is particularly relevant here, showing the three most likely movements alongside a confidence score gives a richer educational experience than a single label.

**Art market and provenance research:** Stylistic period identification is a first step in provenance research for unsigned or disputed works. Narrowing the historical window reduces the expert workload.

**Limitations to acknowledge:**  
The labels in WikiArt are user-submitted and reflect the subjective nature of art historical categorisation where many paintings genuinely belong to more than one movement.
The model produces confident predictions with no uncertainty estimate, which is a meaningful limitation for any real application. Additionally, the model was trained on digital scans; performance on photographs taken in poor lighting or at an angle is unknown.

## 11. Future Work

**Expand to all classes:** The current 10-class selection was a practical compromise. Training on the full WikiArt dataset is the natural next step, and sourcing additional data from open collections (Metropolitan Museum of Art, Rijksmuseum API) could help address class imbalance in underrepresented movements.

**Fine-tuning the transfer learning model:** The current strategy freezes the entire EfficientNetB0 base. 
A next step is to unfreeze the top layers and continue training at a very low learning rate (fine-tuning). 
This allows the pre-trained features to adapt to the specific visual properties of paintings and typically yields additional accuracy gains over feature extraction alone.

**Experiment with other pre-trained architectures:**
- `MobileNetV2` — lighter than EfficientNetB0, interesting for deployment on hardware with limited memory
- `EfficientNetB3` or `B5` — larger variants, suitable if GPU resources are available
- `ResNet50` — classic architecture with residual connections, well-studied in the art classification literature

**Deployment on lower-memory hardware:** EfficientNetB0 is already lightweight however model quantisation (INT8) or knowledge distillation could make the model deployable on mobile devices or museum kiosk hardware without a GPU.

**Multi-label classification:** The current model assigns each painting to exactly one movement. A more realistic formulation would allow multiple labels, reflecting cases where a work genuinely spans two movements — for example, Gauguin's later work sits at the boundary of Post-Impressionism and Primitivism simultaneously.